# Tensorflow multimodal MLP baseline - intermediate fusion on two modalities: EDA+PPG, EDA+TEXT or PPG+TEXT.

Same data, splits, balancing and encoders as `mvae.ipynb`. Each modality encoder gives a 50-d feature, the two features are concatenated inside `MLP` and classified. Trained end-to-end in one stage, no VAE objective.

In [ ]:
import os
os.chdir("/notebooks")

In [ ]:
from lib.install import prepare_env
_ = prepare_env(force=False)

#### 1. Import

In [ ]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
import tensorflow as tf
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.neighbors import NearestNeighbors
from transformers import TFDistilBertModel, DistilBertTokenizerFast
from lib.logger import Logger
from lib.mvae.mvae import Modality
from lib.mlp.mlp import MLP

import tensorflow as tf
import tensorflow_probability as tfp
import tf_keras as tfk
tfd = tfp.distributions


#### 2. Config

In [ ]:
hidden_layers = (100, 50)
latent_dim    = hidden_layers[1]         # equivalent to 2nd hd
seed_values   = [42, 43, 111]
lr            = 1e-3                     # adam learning rate
TASK          = os.getenv("TASK", "AROUSAL")
MODALITY      = os.getenv("MODALITY", "EDA+PPG")
EXPERIMENT    = os.getenv("EXPERIMENT","SIMPLE")
MODEL_NAME    = "MLP"

m1,m2         = MODALITY.split('+')
if m2 == 'TEXT':
    batch_size    = 4
    epochs        = 50
else:
    batch_size    = 200 
    epochs        = 200
    
FSCORE_CALCULATION = 'binary'

DATA_CONFIG = {
    "eda_path": "EEVR/Data_files/EDA_preprocessed_aligned.csv",
    "ppg_path": "EEVR/Data_files/PPG_preprocessed_aligned.csv",
    "text_path": "EEVR/Data_files/TEXT_preprocessed_aligned.csv",
}

#### 3.1 Utility Functions

In [ ]:
# ============================================================
# Utility: CSV read
# ============================================================
def csv_read(path):
    df = pd.read_csv(path)
    return df
        
logger = Logger(MODEL_NAME,MODALITY,TASK,EXPERIMENT)
#can say output dir for writing the models, logger.print would save the results
# in output folder 

# ============================================================
# Utility: seed everything
# ============================================================
def set_seed(seed=42):# default seed value
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed) # different part of codes need different
    np.random.seed(seed) # libraries for randomness
    tf.random.set_seed(seed)
    return seed 

# ============================================================
# Utility: Feature Analysis
# ============================================================
def feature_analysis(df, relevant_features, identifiers):
    df_features = df[relevant_features]
    arousals = df["arousal_category"].tolist()
    num_high_arousal = arousals.count(1)
    num_low_arousal = arousals.count(0)
    #print(df.head(5))
    logger.print(
        f"The Number of entries of High Arousal: {num_high_arousal} "
        f"& Low Arousal: {num_low_arousal}"
    )
    valence = df["valence_category"].tolist()
    num_high_valence = valence.count(1)
    num_low_valence = valence.count(0)

    logger.print(
        f"The Number of entries of High Valence: {num_high_valence} "
        f"& Low Valence: {num_low_valence}"
    )
    taskwiselabel_list = df["taskwiselabel"].tolist()
    num_zeros = taskwiselabel_list.count(0)
    num_ones = taskwiselabel_list.count(1)

    logger.print(
        f"The Number of entries of Positive Task: {num_ones} "
        f"& Negative Task: {num_zeros}"
    )
    pi = df["Participant ID"]
    vi = df["Video ID"]
    df_required = df[relevant_features + identifiers]
    return df_required, pi, vi

# ============================================================
# Utility: metrics
# ============================================================
def compute_metrics(y_true, y_pred, n_classes): # y_true = true label, y_pre=predicted
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    accuracy = accuracy_score(y_true, y_pred)
    if FSCORE_CALCULATION.lower() == "weighted" or n_classes > 2:
        f1score = f1_score(y_true,y_pred,average="weighted",zero_division=0)
    else:
        f1score = f1_score(y_true,y_pred,average="binary",zero_division=0)
    return {"acc": accuracy,"f1": f1score}


# ============================================================
# Utility: SliceDict
# ============================================================
class SliceDict(dict):
    def __getitem__(self,idx):
        if isinstance(idx,str):
            return dict.__getitem__(self,idx)

        return SliceDict({
            k:np.asarray(v)[idx]
            for k,v in self.items()
        })

    def __len__(self):
        return len(next(iter(self.values())))

    @property
    def shape(self):
        return next(iter(self.values())).shape

    def to_matrix(self):
        keys   = list(self.keys())
        arrays = []
        dims   = {}
        for k in keys:
            arr = np.asarray(self[k])
            if arr.ndim == 1:
                arr = arr[:,None]
            dims[k] = arr.shape[1]
            arrays.append(arr)
        X_mat = np.concatenate(arrays,axis=1)
        return X_mat,keys,dims

    @classmethod
    def from_matrix(cls,X_mat,keys,dims):
        X = cls({})
        start = 0

        for k in keys:
            dim = dims[k]
            X[k] = X_mat[:,start:start+dim]
            start += dim
        return X
    
# ============================================================
# Utility: balance data SIMPLE
# Concatenate minority class once
# ============================================================
def balance_data_simple(X_m1_train,X_m2_train,y_train):
    classes,counts = np.unique(y_train,return_counts=True)

    if counts.min() == counts.max():
        return X_m1_train,X_m2_train,y_train

    minority_class = classes[np.argmin(counts)]
    minority_idx   = np.where(y_train==minority_class)[0]

    all_idx = np.concatenate([
        np.arange(len(y_train)),
        minority_idx,
    ],axis=0)

    X_m1_balanced = X_m1_train[all_idx]
    X_m2_balanced = X_m2_train[all_idx]
    y_balanced    = y_train[all_idx]

    return X_m1_balanced,X_m2_balanced,y_balanced

# ============================================================
# Utility: balance data SMOTE
# X_m1_train is always numpy
# X_m2_train is numpy or SliceDict text
# ============================================================
def balance_data_SMOTE(X_m1_train,X_m2_train,y_train,random_state=42):
    classes,counts = np.unique(y_train,return_counts=True)
    minority_count = counts.min()

    if minority_count <= 1:
        sampler = RandomOverSampler(random_state=random_state)
    else:
        sampler = SMOTE(random_state=random_state,k_neighbors=min(5,minority_count-1))

    if isinstance(X_m2_train,SliceDict):
        X_m1_balanced,y_balanced = sampler.fit_resample(X_m1_train,y_train)
        assert np.array_equal(y_balanced[:len(y_train)],y_train), \
                    "Unexpected sampler order: original y_train is not first in y_balanced."

        new_labels = y_balanced[len(y_train):]
        
        rng = np.random.default_rng(random_state)
        add_idx = []

        for lbl in new_labels:
            cls_idx = np.where(y_train==lbl)[0]
            add_idx.append(rng.choice(cls_idx))

        add_idx = np.asarray(add_idx)

        all_idx = np.concatenate([
            np.arange(len(y_train)),
            add_idx,
        ],axis=0)

        X_m2_balanced = X_m2_train[all_idx]

    else:
        dim_m1 = X_m1_train.shape[1]
        dim_m2 = X_m2_train.shape[1]

        X_mat = np.concatenate([X_m1_train,X_m2_train],axis=1)
        X_res,y_balanced = sampler.fit_resample(X_mat,y_train)

        X_m1_balanced = X_res[:,:dim_m1]
        X_m2_balanced = X_res[:,dim_m1:dim_m1+dim_m2]

    return X_m1_balanced,X_m2_balanced,y_balanced


#### 3.2 Utility: Tensorflow model

In [ ]:
# ============================================================
# Encoder
# ============================================================
class Encoder(tfk.Model):
    def __init__(self,input_dim,hidden_layers,name="encoder",**kwargs):
        super().__init__(name=name,**kwargs)

        h1,h2           = hidden_layers
        self.input_dim  = input_dim                                  # input feature dimension
        self.h          = tfk.layers.Dense(h1,activation="relu")      # hidden layer
        self.mu         = tfk.layers.Dense(h2)                        # latent mean
        self.log_var    = tfk.layers.Dense(h2)                        # latent log variance
        self.latent_dim = h2                                          # latent dimension

    def params(self):
        return self.trainable_variables                               # all encoder params

    def call(self,inputs,training=False):
        inputs    = tf.cast(inputs,tf.float32)
        h         = self.h(inputs,training=training)
        z_mean    = self.mu(h)
        z_log_var = self.log_var(h)

        z_log_var = tf.clip_by_value(z_log_var,-8.0,5.0)              # avoid extreme variance
        z_std     = tf.exp(0.5*z_log_var) + 1e-6

        qz_x = tfd.MultivariateNormalDiag(
            loc=z_mean,
            scale_diag=z_std,
        )

        target = inputs                                               # reconstruct original input

        return qz_x,target
    
    
# ============================================================
# Decoder
# ============================================================
class Decoder(tfk.Model):
    def __init__(self,output_dim,hidden_layers,name="decoder",**kwargs):
        super().__init__(name=name,**kwargs)

        h1,_            = hidden_layers
        self.output_dim = output_dim                                  # reconstruction dimension
        self.input_dim  = output_dim                                  # consistent access
        self.h          = tfk.layers.Dense(h1,activation="relu")      # hidden layer
        self.mu         = tfk.layers.Dense(output_dim)                # reconstruction mean
        self.logvar     = tfk.layers.Dense(output_dim)                # reconstruction log variance

    def params(self):
        return self.trainable_variables                               # all decoder params

    def call(self,z,training=False):
        h      = self.h(z,training=training)
        x_mean = self.mu(h)
        x_logv = self.logvar(h)

        x_logv = tf.clip_by_value(x_logv,-8.0,5.0)                    # stable likelihood
        x_std  = tf.exp(0.5*x_logv) + 1e-6

        px_z = tfd.MultivariateNormalDiag(
            loc=x_mean,
            scale_diag=x_std,
        )

        return px_z


# ============================================================
# Text Encoder
# ============================================================
class TextEncoder(tfk.Model):
    def __init__(self,hidden_layers,train_bert=False,name="text_encoder",**kwargs):
        super().__init__(name=name,**kwargs)

        self.bert           = TFDistilBertModel.from_pretrained("distilbert-base-uncased")
        self.bert.trainable = train_bert                              # True trains BERT, False freezes BERT

        self.proj_dim = 256

        self.proj = tfk.Sequential([
            tfk.layers.Dense(self.proj_dim),
            tfk.layers.LayerNormalization(),
            tfk.layers.Activation("relu"),
        ],name="text_projection")                                     # CLS 768 -> 128

        self.encoder = Encoder(
            input_dim=self.proj_dim,
            hidden_layers=hidden_layers,
        )                                                             # VAE encoder on projected CLS

        self.latent_dim = self.encoder.latent_dim
        self.input_dim  = self.proj_dim
        self.train_bert = train_bert

    def bert_params(self):
        if self.bert.trainable:
            return self.bert.trainable_variables                      # BERT params only
        return []                                                     # empty if BERT frozen

    def params(self):
        return (
            self.proj.trainable_variables +
            self.encoder.params() +
            self.bert_params()
        )                                                             # all trainable text encoder params

    def call(self,x,training=False):
        input_ids      = tf.cast(x["input_ids"],tf.int32)
        attention_mask = tf.cast(x["attention_mask"],tf.int32)

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            training=training and self.bert.trainable,
        )                                                             # BERT forward

        last_hidden = outputs.last_hidden_state
        bert_output = last_hidden[:,0,:]                              # CLS embedding

        text_feat = self.proj(bert_output,training=training)          # projected CLS feature

        qz_x,_ = self.encoder(text_feat,training=training)            # q(z|text)

        target = tf.stop_gradient(text_feat)                          # fixed reconstruction target

        return qz_x,target

#### 4. Read Data

In [ ]:
identifiers = [
    "Participant ID",
    "Video ID",
    "arousal_category",
    "valence_category",
    "taskwiselabel",
]

eda_features = [
    "ku_eda", "sk_eda", "dynrange", "slope", "variance",
    "entropy", "insc", "fd_mean", "max_scr", "min_scr",
    "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
]

ppg_features = [
    "BPM", "IBI", "PPG_Rate_Mean", "HRV_MedianNN", "HRV_Prc20NN",
    "HRV_MinNN", "HRV_HTI", "HRV_TINN", "HRV_LF", "HRV_VHF",
    "HRV_LFn", "HRV_HFn", "HRV_LnHF", "HRV_SD1SD2", "HRV_CVI",
    "HRV_PSS", "HRV_PAS", "HRV_PI", "HRV_C1d", "HRV_C1a",
    "HRV_DFA_alpha1", "HRV_MFDFA_alpha1_Width",
    "HRV_MFDFA_alpha1_Peak", "HRV_MFDFA_alpha1_Mean",
    "HRV_MFDFA_alpha1_Max", "HRV_MFDFA_alpha1_Delta",
    "HRV_MFDFA_alpha1_Asymmetry", "HRV_ApEn", "HRV_ShanEn",
    "HRV_FuzzyEn", "HRV_MSEn", "HRV_CMSEn", "HRV_RCMSEn",
    "HRV_CD", "HRV_HFD", "HRV_KFD", "HRV_LZC",
]

text_features = [
    "Text"
]

eda_df = csv_read(DATA_CONFIG["eda_path"])
ppg_df = csv_read(DATA_CONFIG["ppg_path"])
text_df = csv_read(DATA_CONFIG["text_path"])


eda_required, eda_pi, eda_vi = feature_analysis(
    eda_df,
    eda_features,
    identifiers,
)

ppg_required, ppg_pi, ppg_vi = feature_analysis(
    ppg_df,
    ppg_features,
    identifiers,
)

text_required, text_pi, text_vi = feature_analysis(
    text_df,
    text_features,
    identifiers,
)

In [ ]:
# ============================================================
# Create modality inputs separately: X_m1 and X_m2
# ============================================================


tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
texts = text_required[text_features[0]].fillna("").astype(str).tolist()
tokens = tokenizer(texts, padding="max_length", truncation=True,max_length=128,return_tensors="np")
input_ids = tokens["input_ids"].astype("int32")
attention_mask = tokens["attention_mask"].astype("int32")

X_text = SliceDict({
    "input_ids": input_ids,
    "attention_mask": attention_mask,
})
X_eda = np.asarray(eda_required[eda_features]).astype("float32")
X_ppg = np.asarray(ppg_required[ppg_features]).astype("float32")

if MODALITY == "EDA+PPG":
    X_m1 = X_eda
    X_m2 = X_ppg

elif MODALITY == "EDA+TEXT":
    X_m1 = X_eda
    X_m2 = X_text

elif MODALITY == "PPG+TEXT":
    X_m1 = X_ppg
    X_m2 = X_text
else:
    raise ValueError(f"Invalid MODALITY {MODALITY}")
    
# ============================================================
# Select task label
# ============================================================

if TASK == "AROUSAL":
    label_column = "arousal_category"
    n_class = 2

elif TASK == "VALENCE":
    label_column = "valence_category"
    n_class = 2

elif TASK == "STIMULUS-LABEL":
    label_column = "taskwiselabel"
    n_class = 2

else:
    raise ValueError(f"Invalid task {TASK}")


y              = eda_required[label_column].values.astype("int32")
participant_id = eda_pi.values.astype("int32")
video_id       = eda_vi.values.astype("int32")

# ============================================================
# Shape checks
# ============================================================

print("MODALITY:",MODALITY)
print("TASK:",TASK)
print("label column:",label_column)

print("X_m1 shape:",X_m1.shape)
print("X_m2 shape:",X_m2.shape)
print("y shape:",y.shape)
print("Classes:",sorted(set(y)))

### 5. Run LOSO Validation


In [ ]:
import gc
from sklearn.model_selection import LeaveOneGroupOut,GroupShuffleSplit

# ============================================================
# LOSO / GroupShuffle training: two-modality MLP baseline
# ============================================================
performance_rows = []
m1,m2         = MODALITY.split("+")
text_training = True if m2=="TEXT" else False

train_bert = False                                                      # safest first run
bert_lr    = 2e-5                                                     # used only if train_bert=True

for seed in seed_values:
    set_seed(seed)

    if text_training:
        splitter = GroupShuffleSplit(n_splits=2,test_size=0.2,random_state=seed)
        splits   = splitter.split(np.arange(len(y)),y,groups=participant_id)
    else:
        splitter = LeaveOneGroupOut()
        splits   = splitter.split(np.arange(len(y)),y,groups=participant_id)

    for fold_no,(train_idx,test_idx) in enumerate(splits,start=1):
        test_pid = ",".join(map(str,sorted(np.unique(participant_id[test_idx]))))

        tf.keras.backend.clear_session()
        tfk.backend.clear_session()
        gc.collect()

        X_m1_train = X_m1[train_idx]
        X_m2_train = X_m2[train_idx]
        X_m1_test  = X_m1[test_idx]
        X_m2_test  = X_m2[test_idx]

        y_train = y[train_idx]
        y_test  = y[test_idx]

        sample_id = np.char.add(np.char.add(participant_id[test_idx].astype(str),"_"),video_id[test_idx].astype(str))

        # ============================================================
        # Balance training data
        # ============================================================
        if EXPERIMENT=="SMOTE":
            X_m1_train,X_m2_train,y_train = balance_data_SMOTE(X_m1_train,X_m2_train,y_train,random_state=seed)

        elif EXPERIMENT=="SIMPLE":
            X_m1_train,X_m2_train,y_train = balance_data_simple(X_m1_train,X_m2_train,y_train)

        X_train = {"m1":X_m1_train,"m2":X_m2_train}
        X_test  = {"m1":X_m1_test,"m2":X_m2_test}

        dataset = tf.data.Dataset.from_tensor_slices((X_train,y_train)).shuffle(len(y_train),seed=seed).batch(batch_size)

        # ============================================================
        # Fresh model objects for this fold
        # ============================================================
        m1_encoder = Encoder(input_dim=X_m1_train.shape[1],hidden_layers=hidden_layers,name=f"{m1}_encoder")
        m1_decoder = Decoder(output_dim=m1_encoder.input_dim,hidden_layers=hidden_layers,name=f"{m1}_decoder")

        if text_training:
            m2_encoder = TextEncoder(hidden_layers=hidden_layers,train_bert=train_bert,name="TEXT_encoder")
        else:
            m2_encoder = Encoder(input_dim=X_m2_train.shape[1],hidden_layers=hidden_layers,name=f"{m2}_encoder")

        m2_decoder = Decoder(output_dim=m2_encoder.input_dim,hidden_layers=hidden_layers,name=f"{m2}_decoder")

        m1_modality = Modality(name="m1",encoder=m1_encoder,decoder=m1_decoder)
        m2_modality = Modality(name="m2",encoder=m2_encoder,decoder=m2_decoder)

        # ============================================================
        # Train MLP end-to-end: encoders + concatenation + classifier
        # ============================================================
        classifier = MLP(m1_modality=m1_modality,m2_modality=m2_modality,bert_lr=bert_lr)

        classifier_optimizer = tfk.optimizers.Adam(learning_rate=lr,beta_1=0.9,beta_2=0.999,epsilon=1e-8)

        for epoch in range(epochs):
            for x_batch,y_batch in dataset:
                classifier.train(x_batch,y_batch,classifier_optimizer)

        # ============================================================
        # Evaluation
        # ============================================================
        pred_proba = classifier.predict_proba(X_test).numpy().astype("float32").ravel()
        y_pred     = classifier.predict_class(X_test).numpy().astype("int32").ravel()

        logger.add_predictions(
            seed=seed,
            fold=fold_no,
            pid=test_pid,
            sample_id=sample_id,
            y_true=y_test,
            pred_proba=pred_proba,
        )

        metrics = compute_metrics(y_test,y_pred,n_class)

        performance_rows.append({
            "seed":seed,
            "pid":test_pid,
            "acc":metrics["acc"],
            "f1":metrics["f1"],
        })

        perf_df      = pd.DataFrame(performance_rows)
        seed_mean_df = perf_df.groupby("seed")[["acc","f1"]].mean().reset_index()
        global_mean  = seed_mean_df[["acc","f1"]].mean()
        global_std   = seed_mean_df[["acc","f1"]].std()

        logger.print(f"Running Mean | Seed={seed} | fold={fold_no} | pid={test_pid} | Acc={global_mean['acc']:.4f} | F1={global_mean['f1']:.4f}")

summary_df = pd.DataFrame([{
    "seed":"Mean ± Std",
    "acc":f"{global_mean['acc']:.4f} ± {global_std['acc']:.4f}",
    "f1":f"{global_mean['f1']:.4f} ± {global_std['f1']:.4f}",
}])

seed_summary_df = pd.concat([seed_mean_df,summary_df],axis=0,ignore_index=True)

logger.print("Seed-wise mean results:")
logger.print(seed_summary_df)
logger.save_npz()